# REVE with Subject-Specific Prompt Tokens

Instead of using a single shared `cls_query_token` for attention pooling, we assign each subject their own learnable prompt tokens. During attention pooling, the subject's prompt tokens replace the generic query token, allowing the model to learn subject-specific aggregation strategies.

**Key idea:** Each subject gets `num_prompt` learnable tokens of shape `(1, num_prompt, E)`. During attention pooling, these tokens serve as queries, producing a subject-conditioned representation.

## Loading REVE

In [13]:
from transformers import AutoModel
import torch

model = AutoModel.from_pretrained("brain-bzh/reve-base", trust_remote_code=True, torch_dtype="auto")
pos_bank = AutoModel.from_pretrained("brain-bzh/reve-positions", trust_remote_code=True, torch_dtype="auto")

## Subject-Specific Prompt Wrapper

We wrap the pretrained REVE model with a module that:
1. Freezes the REVE backbone
2. Creates a `ParameterList` of `num_subject` prompt tokens, each of shape `(1, num_prompt, embed_dim)`
3. Overrides attention pooling to use the subject's prompt tokens as queries
4. Adds a classification head on top

In [14]:
import torch.nn as nn
from einops import rearrange


class ReveSubjectPrompt(nn.Module):
    def __init__(self, reve_model, num_subject, num_prompt=1, num_classes=4, dropout=0.1):
        super().__init__()
        self.reve = reve_model
        self.embed_dim = reve_model.embed_dim
        self.num_subject = num_subject
        self.num_prompt = num_prompt

        # Freeze the backbone
        for param in self.reve.parameters():
            param.requires_grad = False

        # Subject-specific prompt tokens: one set per subject
        self.subject_prompt_tokens = nn.ParameterList([
            nn.Parameter(torch.randn(1, num_prompt, self.embed_dim))
            for _ in range(num_subject)
        ])

        # Classification head: num_prompt * embed_dim -> num_classes
        self.classifier = nn.Sequential(
            nn.RMSNorm(num_prompt * self.embed_dim),
            nn.Dropout(dropout),
            nn.Linear(num_prompt * self.embed_dim, num_classes),
        )

    def subject_attention_pooling(self, x, subject_ids):
        """
        Attention pooling using subject-specific prompt tokens as queries.

        Args:
            x: (B, C, S, E) output from the REVE backbone
            subject_ids: (B,) integer tensor with subject index for each sample
        Returns:
            (B, num_prompt * E) pooled representation
        """
        b, c, s, e = x.shape
        x = rearrange(x, "b c s e -> b (c s) e")  # (B, C*S, E)

        # Gather subject-specific prompt tokens for each sample in the batch
        # Each prompt token is (1, num_prompt, E), we need (B, num_prompt, E)
        queries = torch.stack([
            self.subject_prompt_tokens[sid.item()] for sid in subject_ids
        ]).squeeze(1)  # (B, num_prompt, E)

        # Scaled dot-product attention
        attention_scores = torch.matmul(queries, x.transpose(-1, -2)) / (self.embed_dim ** 0.5)  # (B, num_prompt, C*S)
        attention_weights = torch.softmax(attention_scores, dim=-1)  # (B, num_prompt, C*S)
        out = torch.matmul(attention_weights, x)  # (B, num_prompt, E)

        return out.reshape(b, -1)  # (B, num_prompt * E)

    def forward(self, eeg, pos, subject_ids):
        # Get backbone features (frozen)
        with torch.no_grad():
            x = self.reve(eeg, pos)  # (B, C, S, E)

        # Subject-conditioned attention pooling
        pooled = self.subject_attention_pooling(x, subject_ids)  # (B, num_prompt * E)

        # Classification
        return self.classifier(pooled)

In [15]:
# Training parameters
num_subject = 9  # BCI IV-2a has 9 subjects
num_prompt = 4   # number of prompt tokens per subject
batch_size = 64
n_epochs = 5
lr_prompt = 1e-3    # learning rate for subject prompt tokens
lr_head = 1e-4      # learning rate for classification head
positions = pos_bank(["Fz", "FC3", "FC1", "FCz", "FC2", "FC4", "C5", "C3", "C1", "Cz", "C2", "C4", "C6", "CP3", "CP1", "CPz", "CP2", "CP4", "P1", "Pz", "P2", "POz"])

# Build the model
prompt_model = ReveSubjectPrompt(model, num_subject=num_subject, num_prompt=num_prompt, num_classes=4)
print(f"Trainable parameters: {sum(p.numel() for p in prompt_model.parameters() if p.requires_grad):,}")
print(f"  - Subject prompt tokens: {num_subject} x ({num_prompt} x {model.embed_dim}) = {num_subject * num_prompt * model.embed_dim:,}")
print(f"  - Classifier parameters: {sum(p.numel() for p in prompt_model.classifier.parameters()):,}")

Trainable parameters: 28,676
  - Subject prompt tokens: 9 x (4 x 512) = 18,432
  - Classifier parameters: 10,244


In [16]:
from transformers import set_seed

set_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Data Loading

We load BCI IV-2a via MOABB. The key difference from the baseline: we extract `subject_id` from the metadata and include it in each sample, so the model knows which subject's prompt tokens to use.

In [17]:
from functools import partial
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
from scipy.signal import butter, lfilter
import numpy as np
from torch.utils.data import Dataset, random_split

def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    return butter(order, [low, high], btype="band")

paradigm = MotorImagery(n_classes=4, resample=250, fmin=8, fmax=30)
bci_dataset = BNCI2014_001()
X, y, metadata = paradigm.get_data(dataset=bci_dataset)

b, a = butter_bandpass(8, 30, 250)
X = lfilter(b, a, X, axis=-1)

label_map = {"left_hand": 0, "right_hand": 1, "feet": 2, "tongue": 3}
y = np.array([label_map[label] for label in y])

# Map subject IDs to 0-indexed integers
unique_subjects = sorted(metadata["subject"].unique())
subject_to_idx = {s: i for i, s in enumerate(unique_subjects)}
subject_ids = np.array([subject_to_idx[s] for s in metadata["subject"]])
print(f"Subjects: {unique_subjects} -> indices {list(subject_to_idx.values())}")


class BCIDatasetWithSubject(Dataset):
    def __init__(self, X, y, subject_ids):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.subject_ids = torch.tensor(subject_ids, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {"data": self.X[idx], "labels": self.y[idx], "subject_id": self.subject_ids[idx]}


n = len(y)
n_train = int(0.7 * n)
n_val = int(0.15 * n)
n_test = n - n_train - n_val
full_dataset = BCIDatasetWithSubject(X, y, subject_ids)
train_ds, val_ds, test_ds = random_split(full_dataset, [n_train, n_val, n_test])


def collate(batch, positions):
    x_data = torch.stack([x["data"] for x in batch])
    y_label = torch.tensor([x["labels"] for x in batch])
    sid = torch.tensor([x["subject_id"] for x in batch])
    positions = positions.repeat(len(batch), 1, 1)
    return {"sample": x_data, "label": y_label.long(), "pos": positions, "subject_id": sid}


collate_fn = partial(collate, positions=positions)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

Choosing from all possible events


Subjects: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)] -> indices [0, 1, 2, 3, 4, 5, 6, 7, 8]


## Training Functions

The training loop now passes `subject_id` to the model so it can select the correct prompt tokens.

In [18]:
from tqdm.auto import tqdm
from sklearn.metrics import balanced_accuracy_score, cohen_kappa_score, f1_score, roc_auc_score, average_precision_score
from sklearn.preprocessing import label_binarize


def train_one_epoch(model, optimizer, loader):
    model.train()
    pbar = tqdm(loader, desc="Training", total=len(loader))

    for batch_data in pbar:
        data, target, pos, subject_id = (
            batch_data["sample"].to(device, non_blocking=True),
            batch_data["label"].to(device, non_blocking=True),
            batch_data["pos"].to(device, non_blocking=True),
            batch_data["subject_id"].to(device, non_blocking=True),
        )
        optimizer.zero_grad()
        with torch.amp.autocast(dtype=torch.float16, device_type="cuda" if torch.cuda.is_available() else "cpu"):
            print(subject_id)
            output = model(data, pos, subject_id)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        pbar.set_postfix({"loss": loss.item()})


def eval_model(model, loader):
    model.eval()

    y_decisions = []
    y_targets = []
    y_probs = []
    score, count = 0, 0
    pbar = tqdm(loader, desc="Evaluating", total=len(loader))
    with torch.inference_mode():
        for batch_data in pbar:
            data, target, pos, subject_id = (
                batch_data["sample"].to(device, non_blocking=True),
                batch_data["label"].to(device, non_blocking=True),
                batch_data["pos"].to(device, non_blocking=True),
                batch_data["subject_id"].to(device, non_blocking=True),
            )
            with torch.amp.autocast(
                dtype=torch.float16, device_type="cuda" if torch.cuda.is_available() else "cpu"
            ):
                output = model(data, pos, subject_id)

            decisions = torch.argmax(output, dim=1)
            score += (decisions == target).int().sum().item()
            count += target.shape[0]
            y_decisions.append(decisions)
            y_targets.append(target)
            y_probs.append(output)

    gt = torch.cat(y_targets).cpu().numpy()
    pr = torch.cat(y_decisions).cpu().numpy()
    pr_probs = torch.softmax(torch.cat(y_probs).float(), dim=1).cpu().numpy()
    acc = score / count
    balanced_acc = balanced_accuracy_score(gt, pr)
    cohen_kappa = cohen_kappa_score(gt, pr)
    f1 = f1_score(gt, pr, average="weighted")

    auroc = roc_auc_score(gt, pr_probs, multi_class='ovr')
    auc_pr = average_precision_score(label_binarize(gt, classes=[0, 1, 2, 3]), pr_probs, average='macro')

    return acc, balanced_acc, cohen_kappa, f1, auroc, auc_pr

## Hyperparameter Search

Grid search over `lr_prompt` and `lr_head` to find the best combination. Each configuration is trained for a few epochs and evaluated on the validation set.

In [ ]:
from itertools import product
from copy import deepcopy

# Grid search parameters
lr_prompt_candidates = [5e-3, 1e-3, 5e-4, 1e-4]
lr_head_candidates = [1e-3, 5e-4, 1e-4, 5e-5]
search_epochs = 10  # enough epochs to get a signal

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = torch.nn.CrossEntropyLoss()

results = []

for lr_p, lr_h in product(lr_prompt_candidates, lr_head_candidates):
    print(f"\n{'='*60}")
    print(f"lr_prompt={lr_p:.0e}, lr_head={lr_h:.0e}")
    print(f"{'='*60}")

    # Fresh model for each config
    search_model = ReveSubjectPrompt(model, num_subject=num_subject, num_prompt=num_prompt, num_classes=4).to(device)

    opt = torch.optim.AdamW([
        {"params": list(search_model.subject_prompt_tokens.parameters()), "lr": lr_p},
        {"params": list(search_model.classifier.parameters()), "lr": lr_h},
    ])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=3)

    best_b_acc = 0
    for epoch in range(search_epochs):
        train_one_epoch(search_model, opt, train_loader)
        _, b_acc, _, _, _, _ = eval_model(search_model, val_loader)
        if b_acc > best_b_acc:
            best_b_acc = b_acc
        sched.step(b_acc)
        print(f"  Epoch {epoch+1}/{search_epochs} — val balanced acc: {b_acc:.4f} (best: {best_b_acc:.4f})")

    results.append({"lr_prompt": lr_p, "lr_head": lr_h, "best_val_bacc": best_b_acc})
    print(f"  >> Best val balanced acc: {best_b_acc:.4f}")

    del search_model, opt, sched
    torch.cuda.empty_cache()

# Sort and display results
results.sort(key=lambda x: x["best_val_bacc"], reverse=True)
print(f"\n{'='*60}")
print("GRID SEARCH RESULTS (sorted by best val balanced accuracy)")
print(f"{'='*60}")
for i, r in enumerate(results):
    marker = " <-- BEST" if i == 0 else ""
    print(f"  lr_prompt={r['lr_prompt']:.0e}, lr_head={r['lr_head']:.0e} => {r['best_val_bacc']:.4f}{marker}")

best_lr_prompt = results[0]["lr_prompt"]
best_lr_head = results[0]["lr_head"]
print(f"\nSelected: lr_prompt={best_lr_prompt:.0e}, lr_head={best_lr_head:.0e}")


lr_prompt=5e-03, lr_head=1e-03


Training:   0%|          | 0/57 [00:00<?, ?it/s]

tensor([7, 6, 8, 4, 0, 4, 0, 8, 4, 5, 0, 4, 2, 0, 1, 6, 3, 0, 2, 3, 1, 7, 5, 7,
        1, 3, 2, 4, 5, 0, 5, 8, 7, 8, 8, 8, 7, 8, 1, 6, 3, 2, 7, 4, 0, 5, 8, 1,
        6, 5, 0, 6, 8, 4, 2, 0, 1, 0, 7, 2, 6, 8, 8, 4], device='cuda:0')
tensor([2, 7, 4, 3, 5, 7, 7, 1, 5, 8, 3, 1, 2, 8, 7, 8, 0, 5, 2, 4, 6, 7, 6, 6,
        5, 5, 7, 3, 6, 8, 4, 7, 6, 0, 0, 5, 6, 2, 3, 3, 4, 6, 0, 3, 5, 5, 7, 7,
        7, 2, 8, 1, 4, 3, 2, 5, 7, 0, 5, 0, 4, 1, 8, 1], device='cuda:0')
tensor([6, 0, 8, 2, 7, 2, 0, 8, 7, 1, 0, 3, 4, 6, 8, 6, 6, 3, 3, 3, 3, 0, 3, 4,
        2, 7, 3, 4, 6, 2, 1, 4, 8, 1, 7, 8, 6, 5, 6, 1, 3, 4, 6, 8, 4, 5, 1, 5,
        3, 0, 5, 3, 6, 6, 6, 2, 0, 7, 3, 8, 8, 8, 3, 3], device='cuda:0')
tensor([3, 4, 2, 0, 3, 3, 5, 5, 6, 7, 6, 6, 4, 5, 6, 2, 2, 8, 0, 0, 4, 6, 4, 7,
        6, 7, 3, 1, 1, 2, 7, 8, 8, 1, 3, 7, 2, 2, 2, 2, 6, 4, 0, 4, 1, 2, 0, 8,
        4, 8, 4, 1, 2, 3, 7, 7, 2, 1, 8, 7, 7, 3, 2, 5], device='cuda:0')
tensor([3, 0, 3, 7, 5, 8, 3, 2, 5, 1, 3, 2, 0, 6, 3, 5, 6, 8, 0,

Evaluating:   0%|          | 0/13 [00:00<?, ?it/s]

  Epoch 1/10 — val balanced acc: 0.2976 (best: 0.2976)


...

Evaluating:   0%|          | 0/13 [00:00<?, ?it/s]

  Epoch 8/10 — val balanced acc: 0.3627 (best: 0.3671)


Training:   0%|          | 0/57 [00:00<?, ?it/s]

tensor([1, 0, 0, 6, 0, 5, 5, 5, 8, 1, 7, 4, 4, 5, 2, 0, 8, 7, 5, 8, 8, 6, 0, 5,
        3, 1, 2, 6, 5, 8, 8, 3, 7, 1, 7, 5, 4, 8, 1, 7, 1, 0, 8, 7, 3, 4, 0, 1,
        8, 0, 6, 8, 5, 5, 3, 1, 2, 8, 5, 0, 1, 2, 1, 7], device='cuda:0')
tensor([2, 2, 5, 4, 6, 8, 1, 2, 0, 3, 6, 4, 1, 4, 3, 5, 2, 6, 6, 4, 7, 7, 5, 3,
        5, 5, 0, 6, 4, 2, 4, 2, 7, 4, 4, 2, 8, 8, 7, 4, 8, 3, 5, 3, 8, 5, 0, 4,
        5, 8, 6, 8, 0, 0, 8, 1, 0, 1, 8, 3, 3, 5, 4, 8], device='cuda:0')
tensor([3, 6, 5, 0, 0, 3, 4, 8, 4, 7, 2, 8, 7, 7, 6, 1, 1, 4, 0, 3, 8, 6, 6, 1,
        4, 0, 7, 1, 0, 4, 2, 0, 8, 5, 3, 7, 7, 0, 5, 4, 4, 6, 2, 1, 5, 4, 8, 0,
        2, 1, 8, 5, 3, 6, 4, 6, 7, 6, 2, 6, 3, 4, 3, 2], device='cuda:0')
tensor([1, 0, 7, 8, 8, 3, 1, 1, 7, 8, 8, 7, 0, 4, 5, 6, 3, 0, 3, 3, 7, 4, 6, 3,
        6, 8, 8, 1, 1, 0, 0, 1, 6, 4, 0, 1, 6, 1, 8, 5, 8, 0, 2, 6, 0, 1, 4, 8,
        7, 8, 1, 0, 7, 4, 4, 5, 5, 5, 8, 3, 0, 3, 0, 4], device='cuda:0')
tensor([2, 6, 5, 7, 7, 8, 3, 2, 8, 0, 7, 0, 8, 5, 1, 2, 4, 2, 3,

Evaluating:   0%|          | 0/13 [00:00<?, ?it/s]

## Final Training

Train with the best hyperparameters found by the grid search.

In [ ]:
print(f"Training with lr_prompt={best_lr_prompt:.0e}, lr_head={best_lr_head:.0e}")

prompt_model = ReveSubjectPrompt(model, num_subject=num_subject, num_prompt=num_prompt, num_classes=4).to(device)

optimizer = torch.optim.AdamW([
    {"params": list(prompt_model.subject_prompt_tokens.parameters()), "lr": best_lr_prompt},
    {"params": list(prompt_model.classifier.parameters()), "lr": best_lr_head},
])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

best_val_acc = 0
best_model_state = None

for epoch in range(n_epochs):
    print(f"Epoch {epoch + 1}/{n_epochs}")
    train_one_epoch(prompt_model, optimizer, train_loader)
    _, b_acc, _, _, _, _ = eval_model(prompt_model, val_loader)
    if b_acc > best_val_acc:
        best_val_acc = b_acc
        best_model_state = {k: v.clone() for k, v in prompt_model.state_dict().items() if v.requires_grad or "subject_prompt" in k or "classifier" in k}
    print(f"Validation balanced accuracy: {b_acc:.4f}, best: {best_val_acc:.4f}")
    scheduler.step(b_acc)

# Reload best model
prompt_model.load_state_dict(best_model_state, strict=False)
acc, balanced_acc, cohen_kappa, f1, auroc, auc_pr = eval_model(prompt_model, test_loader)

print("\n--- Test Results ---")
print(f"Accuracy:          {acc:.4f}")
print(f"Balanced Accuracy: {balanced_acc:.4f}")
print(f"Cohen's Kappa:     {cohen_kappa:.4f}")
print(f"F1 (weighted):     {f1:.4f}")
print(f"AUROC:             {auroc:.4f}")
print(f"AUC-PR:            {auc_pr:.4f}")